# Análisis Exploratorio del Dataset de Productos Cosméticos

Este notebook explora el dataset crudo de productos de cosmética, identifica valores faltantes ocultos y propone un flujo de limpieza y análisis adaptado al contexto del proyecto.

In [ ]:
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

In [ ]:
# Detectar si estamos en Google Colab y cargar el archivo de datos
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    print('Google Colab detectado. Sube `products.csv` cuando se te pida.')
    from google.colab import files
    uploaded = files.upload()
    if uploaded:
        ruta = list(uploaded.keys())[0]
    else:
        raise FileNotFoundError('Debe subir el archivo `products.csv` en Colab antes de ejecutar el notebook.')
else:
    ruta = '../data/raw/products.csv'

print(f'Usando ruta de datos: {ruta}')
df = pd.read_csv(ruta, na_values=['NULL', 'null', ''])

print(f'Dimensiones del dataset: {df.shape}')
df.head()

## Diccionario de variables clave

- `ID`: Identificador único del producto.
- `brand`: Marca del producto.
- `name`: Nombre comercial del producto.
- `product_type`: Tipo de producto (por ejemplo, tratamiento, crema, color de uñas).
- `img`: URL de la imagen principal del producto.
- `rating`: Valoración media del producto.
- `dupes`: Productos similares o duplicados.
- `description`: Descripción del producto.
- `shade_img`: JSON con tonos e imágenes de los tonos.
- `price_site`: Precio en el sitio web.
- `view_count`: Número de vistas del producto.

In [ ]:
# Información general del dataset
print('Información del dataset:')
df.info()

print('\nResumen estadístico de variables numéricas:')
df.describe(include='all').T

In [ ]:
# Análisis de valores faltantes y valores ocultos
missing_counts = df.isna().sum().sort_values(ascending=False)
missing_ratio = (df.isna().mean() * 100).round(2)
missing_summary = pd.DataFrame({'missing_count': missing_counts, 'missing_ratio': missing_ratio})

print('--- Valores faltantes por columna ---')
print(missing_summary[missing_summary['missing_count'] > 0])

plt.figure(figsize=(12, 4))
sns.heatmap(df.isna(), cbar=False, cmap='viridis', yticklabels=False)
plt.title('Mapa de calor de valores faltantes')
plt.xlabel('Columnas')
plt.show()

In [ ]:
# Explorar categorías principales
print('Top 12 tipos de producto:')
print(df['product_type'].value_counts().head(12))

plt.figure(figsize=(12, 5))
ax = sns.countplot(data=df, y='product_type', order=df['product_type'].value_counts().head(12).index, palette='viridis')
ax.set_title('Top 12 tipos de producto')
ax.set_xlabel('Cantidad')
ax.set_ylabel('Tipo de producto')
plt.tight_layout()
plt.show()

print('Top 10 marcas:')
print(df['brand'].value_counts().head(10))

plt.figure(figsize=(12, 5))
ax = sns.countplot(data=df, y='brand', order=df['brand'].value_counts().head(10).index, palette='magma')
ax.set_title('Top 10 marcas con más productos')
ax.set_xlabel('Cantidad de productos')
ax.set_ylabel('Marca')
plt.tight_layout()
plt.show()

In [ ]:
# Análisis de variables numéricas
numeric_cols = df.select_dtypes(include=['number']).columns.tolist()
numeric_cols = [col for col in numeric_cols if col != 'ID']

print('Variables numéricas detectadas:', numeric_cols)

if numeric_cols:
    n_numeric = len(numeric_cols)
    fig, axs = plt.subplots(n_numeric, 1, figsize=(12, 5 * n_numeric), squeeze=False)
    axs = axs.flatten()
    for ax, col in zip(axs, numeric_cols):
        sns.histplot(data=df, x=col, kde=True, ax=ax, color='steelblue')
        ax.set_title(f'Distribución de {col}')
        ax.set_xlabel(col)
        ax.set_ylabel('Frecuencia')
    plt.tight_layout()
    plt.show()
else:
    print('No se encontraron variables numéricas válidas para el análisis.')

In [ ]:
# Identificación de outliers mediante IQR
cols_outliers = [col for col in numeric_cols if col not in ['ID']]
df_capped = df.copy()
for col in cols_outliers:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    df_capped[col] = df[col].clip(lower, upper)

print('Outliers tratados mediante capping para variables numéricas sin perder registros.')

if cols_outliers:
    n_outliers = len(cols_outliers)
    fig, axs = plt.subplots(n_outliers, 2, figsize=(14, 5 * n_outliers), squeeze=False)
    for idx, col in enumerate(cols_outliers):
        sns.boxplot(data=df, x=col, ax=axs[idx, 0], color='indianred')
        axs[idx, 0].set_title(f'Boxplot original de {col}')

        sns.boxplot(data=df_capped, x=col, ax=axs[idx, 1], color='mediumseagreen')
        axs[idx, 1].set_title(f'Boxplot con capping de {col}')

    plt.tight_layout()
    plt.show()
else:
    print('No hay columnas numéricas disponibles para análisis de outliers.')

In [ ]:
# Correlación entre las variables numéricas
if numeric_cols:
    corr_matrix = df[numeric_cols].corr()
    if corr_matrix.empty:
        print('No hay suficientes variables numéricas para calcular la matriz de correlación.')
    else:
        mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
        fig, ax = plt.subplots(figsize=(10, 8))
        sns.heatmap(corr_matrix, mask=mask, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5, vmin=-1, vmax=1, ax=ax)
        ax.set_title('Matriz de correlación de variables numéricas')
        ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
        ax.set_yticklabels(ax.get_yticklabels(), rotation=0)
        plt.tight_layout()
        plt.show()
else:
    print('No se encontraron variables numéricas para la matriz de correlación.')

## Conclusiones preliminares

- El dataset contiene columnas de texto, URL de imágenes y campos numéricos como `price_site`, `view_count` y `rating`.
- Los valores `NULL` se han convertido en `NaN` para facilitar el análisis.
- Los campos `product_type` y `brand` son útiles para análisis de segmentación y pueden requerir limpieza de categorías.
- En la fase de limpieza debemos tratar los valores faltantes en `rating`, `product_type` y `name`, y definir si `shade_img` se mantiene como JSON o se transforma en variables adicionales.
- El siguiente paso es construir un pipeline de transformación con limpieza de texto, normalización de precios y extracción de atributos clave desde `description` y `shade_img`.